# CS 1090B / DS2 — Milestone 4: Artist Authorship Attribution

**Group 166** — Yiqiao Huang, Lixuan Wei, Ruyi Yang, Tian Xia

**Task**: 50-class artist attribution from painting images, with LoRA-based synthetic data augmentation for tail classes.

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Data Preparation](#2-data-preparation)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Baseline Classification](#4-baseline-classification)
5. [LoRA-Based Data Augmentation](#5-lora-based-data-augmentation)
6. [Augmented Classification Experiments](#6-augmented-classification-experiments)
7. [Results & Discussion](#7-results--discussion)

---

### Notes for reviewers

- **No GPU required.** All classifier checkpoints and pre-generated images are downloaded from Google Drive in §1.3. The notebook could run end-to-end on a CPU-only instance.
- **Reproducibility**: *Restart kernel + Run All* on a fresh runtime — no manual setup needed.
- To retrain from scratch (~10 min each on a T4 GPU), set `RETRAIN_BASELINE = True` or `RETRAIN_BP = True` in §4 / §6.

### AI Usage Disclosure

This notebook was developed with assistance from Claude (Anthropic).
AI assistance included:
- debugging Python code
- explaining algorithms
- improving documentation/comments


## 1. Environment Setup

In [1]:
# 1.1 Install dependencies (classification only — no diffusers/controlnet needed to run this notebook)
%pip install --quiet torch torchvision
%pip install --quiet scikit-learn pandas matplotlib seaborn tqdm Pillow gdown

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cpu


In [3]:
# 1.3 Paths
# On Colab: assets are downloaded to /content/project/
# On local: assets are downloaded next to this notebook
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    PROJECT_DIR = Path('/content/project')
else:
    PROJECT_DIR = Path('.').resolve()

DATA_ROOT = PROJECT_DIR / 'dataset'
LORA_DIR  = PROJECT_DIR / 'generated_lora'
CANNY_DIR = PROJECT_DIR / 'generated_lora_canny'
OUT_DIR   = PROJECT_DIR / 'checkpoints'
FIG_DIR   = PROJECT_DIR / 'fig'

CSV_PATH = DATA_ROOT / 'artists.csv'
IMG_DIR  = DATA_ROOT / 'resized' / 'resized'

for d in [OUT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_DIR: {PROJECT_DIR}')

PROJECT_DIR: /Users/ruyiyang/Documents/courseware/2026_Spring/ds2/ds2_project/209b-ArtistAuthorship


In [4]:
# 1.4 Download all assets from Google Drive
# Each asset is skipped if its destination already contains files.
import gdown

DRIVE_DATASET_URL   = 'https://drive.google.com/drive/folders/1JGdPWUOz6tms4MUWaiGJCNfFiNYx9G15'
DRIVE_LORA_URL      = 'https://drive.google.com/drive/folders/1l9Gn_v0vcHLlWXGybwgBqU7qCOfCxk5Y'
DRIVE_BASELINE_URL  = 'https://drive.google.com/drive/folders/1oeMlDCjgi1Gi1lE2fOQDr34iRYI1XrjQ'
DRIVE_AUGMENTED_URL = 'https://drive.google.com/drive/folders/1c5fLwr_KtLCRFnnj7JJu5eR_b4KNBFPd'

def maybe_download(url, dest, label):
    """Download a Drive folder to dest; skip if dest is non-empty."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    if any(dest.iterdir()):
        print(f'[skip]     {label}')
        return
    print(f'[download] {label} → {dest}')
    gdown.download_folder(url, output=str(dest), quiet=True, remaining_ok=True)

def maybe_download_ckpt(url, dest_dir, subfolder, label):
    """Download checkpoint files into dest_dir/subfolder, then move .pth files up to dest_dir.
    Uses a subfolder so two separate checkpoint folders don't block each other's skip check.
    """
    dest_dir = Path(dest_dir)
    staging  = dest_dir / subfolder
    staging.mkdir(parents=True, exist_ok=True)
    if any(staging.iterdir()):
        print(f'[skip]     {label}')
    else:
        print(f'[download] {label} → {staging}')
        gdown.download_folder(url, output=str(staging), quiet=True, remaining_ok=True)
    # Flatten: move any newly downloaded .pth files into dest_dir
    for f in staging.glob('*.pth'):
        target = dest_dir / f.name
        if not target.exists():
            f.rename(target)

maybe_download(DRIVE_DATASET_URL,                        DATA_ROOT,  'Dataset')
maybe_download(DRIVE_LORA_URL + '/generated_lora',       LORA_DIR,   'Generated images (text2img)')
maybe_download(DRIVE_LORA_URL + '/generated_lora_canny', CANNY_DIR,  'Generated images (ControlNet+Canny)')
maybe_download_ckpt(DRIVE_BASELINE_URL,  OUT_DIR, '_baseline',  'Baseline classifier checkpoint')
maybe_download_ckpt(DRIVE_AUGMENTED_URL, OUT_DIR, '_augmented', 'Augmented classifier checkpoint')
print('\nAll assets ready.')

Local environment detected — skipping downloads.
Expected data at: /Users/ruyiyang/Documents/courseware/2026_Spring/ds2/ds2_project/209b-ArtistAuthorship


## 2. Data Preparation

The dataset is [Best Artworks of All Time](https://www.kaggle.com/datasets/ikarus777/best-artworks-of-all-time) (Kaggle): 50 artists, 8,355 images.

We apply a **head / tail split**: artists with fewer than 200 images are considered *tail* classes — these are the targets for synthetic augmentation later.

In [ ]:
df = pd.read_csv(CSV_PATH)
df['name_key'] = df['name'].str.replace(' ', '_')
print(f'Artists in CSV: {len(df)}')
df[['name', 'genre', 'nationality', 'paintings']].head()

In [ ]:
# Collect all image paths and derive labels from filenames (format: ArtistName_NNN.jpg)
img_files = sorted(f for f in os.listdir(IMG_DIR) if f.endswith('.jpg'))
all_paths, all_labels = [], []
for fname in img_files:
    # Strip trailing image index to recover artist name key
    name_key = '_'.join(fname.replace('.jpg', '').split('_')[:-1])
    if name_key in df['name_key'].values:
        all_paths.append(os.path.join(IMG_DIR, fname))
        all_labels.append(name_key)

le = LabelEncoder()
all_labels_enc = le.fit_transform(all_labels)
n_classes = len(le.classes_)
print(f'Total images: {len(all_paths)}, Classes: {n_classes}')

In [ ]:
# Stratified 70 / 15 / 15 train-val-test split
X_tv, X_test, y_tv, y_test = train_test_split(
    all_paths, all_labels_enc, test_size=0.15, stratify=all_labels_enc, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/0.85, stratify=y_tv, random_state=SEED)

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

# Head / tail split (tail = fewer than 200 paintings total)
counts = Counter(all_labels)
TAIL_THRESHOLD = 200
tail_artists = {k for k, v in counts.items() if v < TAIL_THRESHOLD}
head_artists = {k for k, v in counts.items() if v >= TAIL_THRESHOLD}
print(f'Head classes (≥{TAIL_THRESHOLD} imgs): {len(head_artists)}, '
      f'Tail classes (<{TAIL_THRESHOLD} imgs): {len(tail_artists)}')

## 3. Exploratory Data Analysis

Key findings from the full EDA (see [`nb1_eda.ipynb`](nb1_eda.ipynb) for complete analysis):

- **Severe class imbalance**: 24–877 images per artist (36× ratio)
- 26 unique multi-label genres; temporal span 1400–1980
- Color distributions overlap heavily across genres, making color alone insufficient for attribution

In [ ]:
count_series = pd.Series(counts).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['#e07b54' if c in tail_artists else '#5b8db8' for c in count_series.index]
axes[0].bar(range(len(count_series)), count_series.values, color=colors, width=1.0)
axes[0].axhline(TAIL_THRESHOLD, color='red', linestyle='--')
axes[0].set_xlabel('Artist (sorted by count)')
axes[0].set_ylabel('Number of images')
axes[0].set_title('Images per Artist')
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color='#5b8db8', label='Head'),
    Patch(color='#e07b54', label='Tail'),
    plt.Line2D([0], [0], color='red', linestyle='--', label=f'Threshold={TAIL_THRESHOLD}'),
])

genre_counts = Counter()
for genres in df['genre'].dropna():
    for g in genres.split(','):
        genre_counts[g.strip()] += 1
pd.Series(genre_counts).sort_values(ascending=True).tail(15).plot(kind='barh', ax=axes[1], color='#6aaa96')
axes[1].set_title('Top 15 Genres')
axes[1].set_xlabel('Number of artists')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Baseline Classification

We evaluate four pretrained architectures with a frozen backbone + linear head, trained for 10 epochs.
Full experiments are in [`nb2_baseline.ipynb`](nb2_baseline.ipynb); here we reproduce the ResNet-18 run (best baseline) and load results for the others.

| Model | Strategy | Test Acc |
|---|---|---|
| ResNet-18 | Full fine-tune | **0.789** |
| ViT-B/16 | Linear probe | 0.648 |
| CLIP ViT-L/14 | Zero-shot | 0.638 |
| EfficientNet-B0 | Linear probe | 0.560 |

In [ ]:
# ── Shared dataset and transform ─────────────────────────────────────────────
class ArtDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.tf = paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.tf(img), self.labels[idx]


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [ ]:
def build_resnet18_frozen(n_classes):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for imgs, labs in loader:
        imgs, labs = imgs.to(device), labs.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct += (out.argmax(1) == labs).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for imgs, labs in loader:
        out = model(imgs.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        trues.extend(labs.numpy())
    preds, trues = np.array(preds), np.array(trues)
    return preds, trues, (preds == trues).mean()

In [ ]:
# ── Baseline A: load pre-trained checkpoint ──────────────────────────────────
# Set RETRAIN_BASELINE = True to train from scratch instead (~10 min on T4).
# Checkpoint is downloaded from Drive in §5.1.
RETRAIN_BASELINE = False
CKPT_A = OUT_DIR / 'resnet18_baseline.pth'

model_A = build_resnet18_frozen(n_classes).to(device)

if RETRAIN_BASELINE or not CKPT_A.exists():
    label_counts = np.bincount(y_train, minlength=n_classes)
    weights = 1.0 / label_counts[y_train]
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

    train_loader = DataLoader(ArtDataset(X_train, y_train, train_tf),
                              batch_size=64, sampler=sampler, num_workers=2)
    val_loader   = DataLoader(ArtDataset(X_val, y_val, val_tf),
                              batch_size=64, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_A.fc.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

    for epoch in range(1, 11):
        tr_loss, tr_acc = train_one_epoch(model_A, train_loader, criterion, optimizer)
        _, _, val_acc = evaluate(model_A, val_loader)
        scheduler.step()
        print(f'Epoch {epoch:02d}  train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  val_acc={val_acc:.3f}')

    torch.save(model_A.state_dict(), CKPT_A)
    print(f'Saved checkpoint → {CKPT_A}')
else:
    model_A.load_state_dict(torch.load(CKPT_A, map_location=device))
    print(f'Loaded baseline checkpoint from {CKPT_A}')

test_loader = DataLoader(ArtDataset(X_test, y_test, val_tf),
                         batch_size=64, shuffle=False, num_workers=2)
preds_A, true_test, acc_A = evaluate(model_A, test_loader)
print(f'\nBaseline (A) test accuracy: {acc_A:.4f}')

In [ ]:
# Per-class recall for baseline — identify the worst-10 artists
rep_A = classification_report(true_test, preds_A, target_names=le.classes_,
                               output_dict=True, zero_division=0)
recall_A = {c: rep_A[c]['recall'] for c in le.classes_}

worst10 = sorted(recall_A, key=recall_A.get)[:10]
print('Worst-10 artists by recall (baseline):')
for artist in worst10:
    print(f'  {artist:<40} recall={recall_A[artist]:.3f}  '
          f'({counts[artist]} training images)')

## 5. LoRA-Based Data Augmentation

We use **DreamBooth + LoRA** fine-tuning on SDXL to train a per-artist style adapter from the small set of real paintings (~40 images per artist). The adapter is then used to synthesize 200 new images per artist.


Full training and generation scripts are in [`data_generation_inference/`](data_generation_inference/):
- `train_and_generate_colab.ipynb` — DreamBooth-LoRA training notebook
- `Phase1_Augmentation_LoRA_Text2Img.py` — batch text2img generation
- `Phase1_Augmentation_LoRA_ControlNet.py` — batch Canny+ControlNet generation

**This section loads pre-generated images by default.** Set `RUN_GENERATION = True` to trigger generation (requires CUDA with ≥16 GB VRAM and pre-trained LoRA checkpoints).

In [ ]:
RUN_GENERATION = False  # Set True to run generation pipeline (slow, GPU required)

if RUN_GENERATION:
    import subprocess, sys
    script = Path('data_generation_inference/Phase1_Augmentation_LoRA_Text2Img.py')
    # Generate for worst-10 artists only (subset run for demo)
    artists_arg = ' '.join(a.lower() for a in worst10)
    cmd = [
        sys.executable, str(script),
        '--lora-root', str(PROJECT_DIR / 'lora_ckpts'),
        '--out-dir',   str(LORA_DIR),
        '--artists',   *[a.lower() for a in worst10],
        '--steps', '20',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping generation — using pre-generated images.')
    if not LORA_DIR.exists():
        print(f'WARNING: {LORA_DIR} not found. '
              'Download pre-generated images or set RUN_GENERATION=True.')

In [ ]:
# Load synthetic images (worst-10 artists only)
lower_to_namekey = {nk.lower(): nk for nk in df['name_key']}

X_synth, y_synth = [], []
synth_counts = {}

if LORA_DIR.exists():
    for folder in sorted(os.listdir(LORA_DIR)):
        full = LORA_DIR / folder
        if not full.is_dir():
            continue
        name_key = lower_to_namekey.get(folder.lower())
        if name_key is None or name_key not in le.classes_:
            continue
        # Only augment worst-10 artists
        if name_key not in worst10:
            continue
        imgs = sorted(full.glob('gen_*.png'))
        label = le.transform([name_key])[0]
        X_synth.extend(str(p) for p in imgs)
        y_synth.extend([label] * len(imgs))
        synth_counts[name_key] = len(imgs)

y_synth = np.array(y_synth, dtype=int)
print(f'Synthetic images loaded: {len(X_synth)}')
for artist, cnt in synth_counts.items():
    print(f'  {artist:<40} +{cnt} images')

In [ ]:
# Visual sanity check: show sample generated images for each augmented artist
if len(X_synth) > 0:
    n_show = min(len(worst10), 10)
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, artist in zip(axes.flatten()[:n_show], worst10):
        if artist not in synth_counts:
            ax.axis('off')
            continue
        label = le.transform([artist])[0]
        idxs = np.where(y_synth == label)[0]
        img = Image.open(X_synth[idxs[0]]).convert('RGB')
        ax.imshow(img)
        ax.set_title(artist.replace('_', ' '), fontsize=7)
        ax.axis('off')
    plt.suptitle('Sample LoRA-generated images (worst-10 artists)', fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Augmented Classification Experiments

We run three experiments, all using the same ResNet-18 frozen backbone + linear head:

| Experiment | Training data | Sampler |
|---|---|---|
| **A** (baseline) | Real only | Inverse-frequency |
| **B** (naive augment) | Real + synth | Inverse-frequency (includes synth in counts) |
| **B'** (corrected) | Real + synth | Per-class real-majority rule (`REAL_FRAC=0.70`) |

The key finding from `B` is that the standard inverse-frequency sampler causes the model to see ~84% synthetic images for worst-10 classes — it learns LoRA generation style, not real painting style. `B'` fixes this by enforcing that at least `REAL_FRAC` of in-class samples are real.

Full sweep and Canny+ControlNet comparison are in [`nb3_lora_augmented.ipynb`](nb3_lora_augmented.ipynb).

In [ ]:
def make_corrected_sampler(y_train_real, y_synth, n_classes, real_frac=0.70):
    """Build a WeightedRandomSampler that enforces real_frac real images per class.

    For each class the probability assigned to each real sample is scaled so that
    the expected proportion of real samples within the class equals real_frac.
    This prevents the model from over-fitting to synthetic style when synth >> real.
    """
    y_all = np.concatenate([y_train_real, y_synth])
    n_all = len(y_all)
    real_counts = np.bincount(y_train_real, minlength=n_classes).astype(float)
    synth_counts = np.bincount(y_synth, minlength=n_classes).astype(float)

    w = np.zeros(n_all)
    n_real = len(y_train_real)

    for cls in range(n_classes):
        rc, sc = real_counts[cls], synth_counts[cls]
        if rc == 0:
            continue
        # Weight per real sample to achieve real_frac share of this class
        w_real = real_frac / rc
        # Weight per synth sample (complement)
        w_synth = ((1 - real_frac) / sc) if sc > 0 else 0.0

        real_idxs = np.where(y_train_real == cls)[0]
        synth_idxs = np.where(y_synth == cls)[0] + n_real
        w[real_idxs] = w_real
        w[synth_idxs] = w_synth

    return WeightedRandomSampler(w, len(w), replacement=True)

In [ ]:
# ── Experiment B' — load checkpoint or retrain ───────────────────────────────
# Set RETRAIN_BP = True to retrain from scratch (~10 min on T4).
# Checkpoint is downloaded from Drive in §5.1.
REAL_FRAC = 0.70
CKPT_BP   = OUT_DIR / f'resnet18_augmented_rf{int(REAL_FRAC*100)}.pth'
RETRAIN_BP = False

X_train_aug = list(X_train) + list(X_synth)
y_train_aug = np.concatenate([y_train, y_synth])

model_Bp = build_resnet18_frozen(n_classes).to(device)

if RETRAIN_BP or not CKPT_BP.exists():
    sampler_bp = make_corrected_sampler(y_train, y_synth, n_classes, real_frac=REAL_FRAC)
    train_loader_aug = DataLoader(
        ArtDataset(X_train_aug, y_train_aug, train_tf),
        batch_size=64, sampler=sampler_bp, num_workers=2)
    val_loader = DataLoader(ArtDataset(X_val, y_val, val_tf),
                            batch_size=64, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_Bp.fc.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

    for epoch in range(1, 11):
        tr_loss, tr_acc = train_one_epoch(model_Bp, train_loader_aug, criterion, optimizer)
        _, _, val_acc = evaluate(model_Bp, val_loader)
        scheduler.step()
        print(f'Epoch {epoch:02d}  train_loss={tr_loss:.4f}  val_acc={val_acc:.3f}')

    torch.save(model_Bp.state_dict(), CKPT_BP)
    print(f'Saved → {CKPT_BP}')
else:
    model_Bp.load_state_dict(torch.load(CKPT_BP, map_location=device))
    print(f'Loaded augmented checkpoint from {CKPT_BP}')

preds_Bp, _, acc_Bp = evaluate(model_Bp, test_loader)
print(f"\nB' (corrected, real_frac={REAL_FRAC}) test accuracy: {acc_Bp:.4f}")

## 7. Results & Discussion

In [ ]:
rep_Bp   = classification_report(true_test, preds_Bp, target_names=le.classes_,
                                output_dict=True, zero_division=0)
recall_Bp = {c: rep_Bp[c]['recall'] for c in le.classes_}

ordered = sorted(worst10, key=lambda c: recall_A[c])
x, w = np.arange(len(ordered)), 0.38

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, [recall_A[c]  for c in ordered], w, label='A: Baseline',    color='#5b8db8')
ax.bar(x + w/2, [recall_Bp[c] for c in ordered], w, label="B': Corrected",  color='#e07b54')
ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', ' ') for c in ordered], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Recall'); ax.set_ylim(0, 1)
ax.set_title('Per-artist recall: Baseline vs. Corrected augmentation (worst-10 artists)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'recall_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
others = [c for c in le.classes_ if c not in worst10]

def group_recall(recall_dict, group):
    return np.mean([recall_dict[c] for c in group])

summary = pd.DataFrame({
    'Experiment':    ["A: Baseline", "B': Corrected (REAL_FRAC=0.70)"],
    'Overall Acc':   [f'{acc_A:.4f}',  f'{acc_Bp:.4f}'],
    'Worst-10 Recall (avg)': [
        f'{group_recall(recall_A, worst10):.4f}',
        f'{group_recall(recall_Bp, worst10):.4f}',
    ],
    'Other-40 Recall (avg)': [
        f'{group_recall(recall_A, others):.4f}',
        f'{group_recall(recall_Bp, others):.4f}',
    ],
})
print(summary.to_string(index=False))

In [ ]:
# ── Regression check: did non-augmented artists suffer? ──────────────────────
delta_others = sorted(
    [(c, recall_Bp[c] - recall_A[c]) for c in others],
    key=lambda x: x[1]
)
print('Biggest drops in non-augmented classes:')
for artist, delta in delta_others[:5]:
    print(f'  {artist:<40} Δ={delta:+.3f}')

print('\nBiggest gains in non-augmented classes:')
for artist, delta in reversed(delta_others[-5:]):
    print(f'  {artist:<40} Δ={delta:+.3f}')

### Key findings

1. **Naive augmentation (B) hurts** — the standard inverse-frequency sampler causes the worst-10 classes to be drawn ~84% from synthetic images, making the classifier learn LoRA generation artifacts instead of real painting characteristics.

2. **Corrected sampler (B') recovers the gain** — enforcing `REAL_FRAC=0.70` (at least 70% real images per class per batch) yields a meaningful improvement in worst-10 recall with minimal regression on other artists.

3. **Generation method matters less than sampling** — ControlNet+Canny and pure text2img produce similar downstream accuracy when the sampler is corrected (see `nb3_lora_augmented.ipynb §9` for head-to-head comparison).

### Limitations & future work

- LoRA training uses only ~40 real images per artist; more training data would improve generation fidelity.
- The corrected sampler introduces a hyperparameter (`REAL_FRAC`) that may need tuning per dataset.
- Full fine-tuning (unfreezing the backbone) was not explored in the augmented setting and may yield larger gains.